# 🤟 ASL Transformer — Complete Training Notebook

**Run this notebook on Google Colab with a GPU runtime.**

### Before you start
1. `Runtime → Change runtime type → T4 GPU` (free) or `A100` (Pro)
2. Add your Gemini API key in the **Config** cell below (optional — only needed for the demo)
3. Run all cells top-to-bottom (`Runtime → Run all`)

### What this notebook does
| Step | Description | Est. Time |
|---|---|---|
| 1 | Install dependencies | 3-5 min |
| 2 | Clone project + download WLASL dataset | 10-20 min |
| 3 | Run RTMPose preprocessing | 30-90 min |
| 4 | Train Transformer classifier | 2-5 hours |
| 5 | Evaluate + generate thesis figures | 5 min |
| 6 | Save model to Google Drive | 1 min |

## ① Setup & Dependencies

In [ ]:
# Verify GPU is available
import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU found — go to Runtime → Change runtime type → GPU')

In [ ]:
%%capture
# Install MMPose and dependencies
# This takes 3-5 minutes — output is suppressed to keep the notebook clean
import subprocess, sys

# Step 1: mmcv (must match torch + cuda version)
torch_ver = torch.__version__.split('+')[0]   # e.g. '2.1.0'
cu_tag    = 'cu121'                            # Colab default as of 2025
mmcv_url  = f'https://download.openmmlab.com/mmcv/dist/{cu_tag}/torch{torch_ver}/index.html'

subprocess.run([sys.executable, '-m', 'pip', 'install', 'mmcv==2.1.0',
                '-f', mmcv_url, '-q'], check=True)

# Step 2: remaining packages
packages = [
    'mmengine>=0.10.0',
    'mmdet>=3.2.0',
    'mmpose>=1.3.0',
    'google-generativeai',
    'gdown',
    'python-dotenv',
    'scikit-learn',
    'tensorboard',
    'opencv-python-headless',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + packages, check=True)
print('✅ All packages installed')

In [ ]:
# Verify key imports
import mmpose, mmdet, mmengine
print(f'mmpose  : {mmpose.__version__}')
print(f'mmdet   : {mmdet.__version__}')
print(f'mmengine: {mmengine.__version__}')
print('✅ MMPose stack ready')

## ② Clone Project & Configuration

In [ ]:
# ══════════════════════════════════════════════════════
#  CONFIGURATION — edit these before running
# ══════════════════════════════════════════════════════

GITHUB_REPO    = 'https://github.com/YOUR_USERNAME/asl-translator.git'  # ← your repo
GEMINI_API_KEY = ''   # ← paste your free key from https://aistudio.google.com/
SAVE_TO_GDRIVE = True # mount Google Drive to persist checkpoints

# Training hyperparameters (mirrors training/config.py)
NUM_CLASSES    = 100
BATCH_SIZE     = 64
EPOCHS         = 50
LR             = 1e-3
WEIGHT_DECAY   = 1e-4
PATIENCE       = 10    # early stopping
WINDOW_SIZE    = 32    # frames per clip (T)
D_MODEL        = 256
NHEAD          = 8
NUM_LAYERS     = 4
DIM_FF         = 512
DROPOUT        = 0.3

In [ ]:
import os

if SAVE_TO_GDRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    GDRIVE_DIR = '/content/drive/MyDrive/asl-translator'
    os.makedirs(GDRIVE_DIR, exist_ok=True)
    print(f'✅ Google Drive mounted → {GDRIVE_DIR}')
else:
    GDRIVE_DIR = None
    print('⚠️  Not saving to Drive — checkpoints will be lost when Colab disconnects')

In [ ]:
import os

PROJECT_DIR = '/content/asl-translator'

if os.path.exists(PROJECT_DIR):
    print(f'Project already cloned at {PROJECT_DIR}')
    %cd {PROJECT_DIR}
else:
    if 'YOUR_USERNAME' not in GITHUB_REPO:
        !git clone {GITHUB_REPO} {PROJECT_DIR}
        %cd {PROJECT_DIR}
    else:
        # Fallback: create project structure inline if repo not set up yet
        print('⚠️  GITHUB_REPO not set — creating project structure inline')
        os.makedirs(PROJECT_DIR, exist_ok=True)
        %cd {PROJECT_DIR}
        for d in ['data/raw', 'data/keypoints', 'models/checkpoints',
                  'training/logs', 'docs/thesis_figures']:
            os.makedirs(d, exist_ok=True)

print(f'Working directory: {os.getcwd()}')

## ③ Download WLASL Dataset

In [ ]:
# Download WLASL metadata and video clips
# This tries individual video URLs first; falls back to --gdrive if needed

!python data/download_wlasl.py \
    --output-dir data/raw \
    --word-list data/word_list.json \
    --max-per-class 80

# If many URLs are dead, uncomment the line below instead:
# !python data/download_wlasl.py --gdrive --gdrive-key WLASL300

In [ ]:
import json, os
from pathlib import Path

# Count downloaded videos per word
video_dir = Path('data/raw/videos')
if video_dir.exists():
    stats = {d.name: len(list(d.glob('*.mp4'))) for d in sorted(video_dir.iterdir()) if d.is_dir()}
    total = sum(stats.values())
    print(f'Total videos downloaded: {total}')
    print(f'Words with videos: {sum(1 for v in stats.values() if v > 0)}/100')
    print(f'\nPer-word counts:')
    for word, count in sorted(stats.items(), key=lambda x: -x[1])[:20]:
        bar = '█' * min(count, 40)
        print(f'  {word:<20} {bar} {count}')
    if total > 20:
        print(f'  ... and {len(stats)-20} more words')
else:
    print('⚠️  No videos found — check download step above')

## ④ RTMPose Preprocessing

In [ ]:
# Run RTMPose on all videos → save (32, 399) .npy files
# ⏱ This is the longest step: ~30-90 min depending on dataset size and GPU

!python data/preprocess.py \
    --video-dir  data/raw/videos \
    --output-dir data/keypoints

# After it finishes, verify output:
!python data/preprocess.py --verify --output-dir data/keypoints

In [ ]:
# Visualise a sample keypoint tensor to sanity-check preprocessing
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

npy_files = list(Path('data/keypoints').rglob('*.npy'))
if not npy_files:
    print('No .npy files found yet')
else:
    sample = np.load(str(npy_files[0]))   # (32, 399)
    print(f'Sample: {npy_files[0].parent.name}/{npy_files[0].name}')
    print(f'Shape : {sample.shape}  (T=32, features=399)')
    print(f'Range : [{sample.min():.3f}, {sample.max():.3f}]')

    # Plot motion trajectory (wrist x-coordinate over time as a proxy)
    # Right wrist = keypoint index 16, x = feature index 16*3 = 48
    right_wrist_x = sample[:, 48]   # (32,)
    right_wrist_y = sample[:, 49]   # (32,)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(right_wrist_x, label='x', color='steelblue')
    axes[0].plot(right_wrist_y, label='y', color='coral')
    axes[0].set(title='Right wrist trajectory over 32 frames',
                xlabel='Frame', ylabel='Normalised coordinate')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(right_wrist_x, right_wrist_y, 'o-', color='mediumorchid', alpha=0.7)
    axes[1].set(title='Wrist XY path (normalised space)',
                xlabel='x', ylabel='y')
    axes[1].grid(alpha=0.3)
    axes[1].invert_yaxis()

    plt.suptitle(f'Word: {npy_files[0].parent.name}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('docs/thesis_figures/sample_trajectory.png', dpi=150)
    plt.show()
    print('✅ Saved: docs/thesis_figures/sample_trajectory.png')

## ⑤ Model Definition (inline copy for Colab)

In [ ]:
# If running from cloned repo, models/transformer.py is already available.
# Otherwise the cell below defines it inline.

import sys
sys.path.insert(0, '/content/asl-translator')

try:
    from models.transformer import ASLTransformer, build_model
    print('✅ Loaded ASLTransformer from models/transformer.py')
except ImportError:
    print('Defining ASLTransformer inline...')
    import torch, torch.nn as nn, torch.nn.functional as F

    class ASLTransformer(nn.Module):
        def __init__(self, num_classes=100, input_dim=399, seq_len=32,
                     d_model=256, nhead=8, num_layers=4, dim_feedforward=512, dropout=0.3):
            super().__init__()
            self.input_projection  = nn.Linear(input_dim, d_model)
            self.cls_token         = nn.Parameter(torch.zeros(1, 1, d_model))
            self.pos_embedding     = nn.Parameter(torch.zeros(1, seq_len + 1, d_model))
            nn.init.trunc_normal_(self.cls_token,     std=0.02)
            nn.init.trunc_normal_(self.pos_embedding, std=0.02)
            encoder_layer = nn.TransformerEncoderLayer(
                d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
                dropout=dropout, activation='gelu', batch_first=True, norm_first=True)
            self.transformer_encoder = nn.TransformerEncoder(
                encoder_layer, num_layers=num_layers, norm=nn.LayerNorm(d_model))
            self.classifier = nn.Sequential(
                nn.LayerNorm(d_model), nn.Dropout(dropout), nn.Linear(d_model, num_classes))
            for m in self.modules():
                if isinstance(m, nn.Linear):
                    nn.init.xavier_uniform_(m.weight)
                    if m.bias is not None: nn.init.zeros_(m.bias)

        def forward(self, x, padding_mask=None):
            B, T, _ = x.shape
            if padding_mask is None:
                padding_mask = (x.abs().sum(dim=-1) == 0)
            x   = self.input_projection(x)
            cls = self.cls_token.expand(B, -1, -1)
            x   = torch.cat([cls, x], dim=1) + self.pos_embedding
            cls_mask  = torch.zeros(B, 1, dtype=torch.bool, device=x.device)
            full_mask = torch.cat([cls_mask, padding_mask], dim=1)
            x   = self.transformer_encoder(x, src_key_padding_mask=full_mask)
            return self.classifier(x[:, 0, :])

        def count_parameters(self):
            return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def build_model(num_classes=100, device='cpu'):
        m = ASLTransformer(num_classes=num_classes).to(device)
        print(f'[model] params: {m.count_parameters():,}')
        return m

# Quick smoke test
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
model  = build_model(num_classes=NUM_CLASSES, device=DEVICE)
dummy  = torch.randn(4, WINDOW_SIZE, 399).to(DEVICE)
out    = model(dummy)
print(f'Input : {dummy.shape}  →  Output: {out.shape}  (expected [4, {NUM_CLASSES}])')
print(f'✅ Model OK on {DEVICE}')

## ⑥ Dataset & DataLoaders

In [ ]:
import json
import random
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class WLASLDataset(Dataset):
    def __init__(self, manifest_path, split, keypoints_dir, augment=False):
        self.kd  = Path(keypoints_dir)
        self.aug = augment and split == 'train'
        with open(manifest_path) as f:
            m = json.load(f)
        self.w2i = m['word_to_idx']
        self.i2w = {v: k for k, v in self.w2i.items()}
        raw = m['splits'][split]
        self.samples = [s for s in raw
                        if (self.kd / s['gloss'] / f"{s['video_id']}.npy").exists()]
        miss = len(raw) - len(self.samples)
        if miss: print(f'[{split}] {miss} missing npy files skipped')
        print(f'[{split}] {len(self.samples)} samples | {len(self.w2i)} classes')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        s  = self.samples[idx]
        kp = np.load(str(self.kd / s['gloss'] / f"{s['video_id']}.npy"))  # (32,399)
        if self.aug: kp = self._augment(kp)
        return torch.tensor(kp, dtype=torch.float32), s['label']

    def _augment(self, kp):
        kp = kp.copy()
        if random.random() < 0.5:  # temporal jitter
            s = random.randint(-3, 3)
            kp = np.roll(kp, s, axis=0)
            if s > 0: kp[:s] = 0
            elif s < 0: kp[s:] = 0
        if random.random() < 0.5:  # keypoint noise
            n = np.random.normal(0, 0.01, kp.shape).astype(np.float32)
            n[:, 2::3] = 0
            kp += n
        if random.random() < 0.5:  # frame drop
            drop = random.sample(range(32), max(1, int(32 * 0.1)))
            kp[drop] = 0
        if random.random() < 0.5:  # horizontal flip
            kp[:, 0::3] = -kp[:, 0::3]
        return kp

MANIFEST = 'data/raw/manifest.json'
KP_DIR   = 'data/keypoints'
NUM_WORKERS = 2  # keep low on Colab to avoid shared memory issues

train_ds = WLASLDataset(MANIFEST, 'train', KP_DIR, augment=True)
val_ds   = WLASLDataset(MANIFEST, 'val',   KP_DIR)
test_ds  = WLASLDataset(MANIFEST, 'test',  KP_DIR)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

# Peek at one batch
xb, yb = next(iter(train_loader))
print(f'Batch x: {xb.shape}  y: {yb.shape}  y range: [{yb.min()}, {yb.max()}]')
print('✅ DataLoaders ready')

## ⑦ Training

In [ ]:
import time, json
from collections import defaultdict
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

def topk_accuracy(logits, labels, topk=(1, 5)):
    with torch.no_grad():
        maxk  = max(topk)
        _, pred = logits.topk(maxk, dim=1)
        pred  = pred.t()
        correct = pred.eq(labels.view(1, -1).expand_as(pred))
        return {f'top{k}': correct[:k].reshape(-1).float().mean().item() * 100 for k in topk}

def run_epoch(model, loader, criterion, optimizer=None, device='cuda', desc=''):
    training = optimizer is not None
    model.train() if training else model.eval()
    totals = defaultdict(float)
    ctx    = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss   = criterion(logits, y)
            if training:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            accs = topk_accuracy(logits, y)
            totals['loss']  += loss.item()
            totals['top1']  += accs['top1']
            totals['top5']  += accs['top5']
    n = len(loader)
    return {k: v / n for k, v in totals.items()}

print('✅ Training helpers defined')

In [ ]:
import os, shutil

CKPT_PATH = 'models/checkpoints/best_model.pth'
LOG_PATH  = 'training/logs/history.json'
os.makedirs('models/checkpoints', exist_ok=True)
os.makedirs('training/logs',      exist_ok=True)

model     = build_model(num_classes=NUM_CLASSES, device=DEVICE)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS - 3, eta_min=1e-5)

best_val_top1   = 0.0
patience_count  = 0
history         = []

print(f'Training on {DEVICE} | {EPOCHS} epochs | batch={BATCH_SIZE} | lr={LR}')
print('=' * 65)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    tr = run_epoch(model, train_loader, criterion, optimizer, DEVICE)
    va = run_epoch(model, val_loader,   criterion, None,      DEVICE)
    scheduler.step()

    elapsed = time.time() - t0
    lr_now  = optimizer.param_groups[0]['lr']

    marker = ''
    if va['top1'] > best_val_top1:
        best_val_top1  = va['top1']
        patience_count = 0
        torch.save({'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'val_top1': best_val_top1}, CKPT_PATH)
        marker = '  ← best'
        # Mirror to Google Drive
        if GDRIVE_DIR:
            shutil.copy(CKPT_PATH, f'{GDRIVE_DIR}/best_model.pth')
    else:
        patience_count += 1

    history.append({'epoch': epoch, 'train': tr, 'val': va, 'lr': lr_now})

    print(f'Ep {epoch:3d}/{EPOCHS} | '
          f'tr loss {tr["loss"]:.3f} top1 {tr["top1"]:5.1f}% | '
          f'va loss {va["loss"]:.3f} top1 {va["top1"]:5.1f}% top5 {va["top5"]:5.1f}% | '
          f'lr {lr_now:.2e} | {elapsed:.0f}s{marker}')

    if patience_count >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch} (patience={PATIENCE})')
        break

with open(LOG_PATH, 'w') as f:
    json.dump(history, f, indent=2)

print(f'\n✅ Training complete  |  Best val top-1: {best_val_top1:.1f}%')
print(f'   Checkpoint: {CKPT_PATH}')

## ⑧ Training Curves (Thesis Figures)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import json

with open('training/logs/history.json') as f:
    hist = json.load(f)

epochs    = [h['epoch']              for h in hist]
tr_loss   = [h['train']['loss']      for h in hist]
va_loss   = [h['val']['loss']        for h in hist]
tr_top1   = [h['train']['top1']      for h in hist]
va_top1   = [h['val']['top1']        for h in hist]
va_top5   = [h['val']['top5']        for h in hist]
lr_curve  = [h['lr']                 for h in hist]

fig = plt.figure(figsize=(15, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# Plot 1: Loss
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(epochs, tr_loss, label='Train', color='steelblue')
ax1.plot(epochs, va_loss, label='Val',   color='coral', linestyle='--')
ax1.set(title='Cross-Entropy Loss', xlabel='Epoch', ylabel='Loss')
ax1.legend(); ax1.grid(alpha=0.3)

# Plot 2: Top-1 Accuracy
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(epochs, tr_top1, label='Train', color='steelblue')
ax2.plot(epochs, va_top1, label='Val',   color='coral', linestyle='--')
ax2.axhline(max(va_top1), color='green', linestyle=':', alpha=0.7,
             label=f'Best: {max(va_top1):.1f}%')
ax2.set(title='Top-1 Accuracy', xlabel='Epoch', ylabel='Accuracy (%)')
ax2.legend(); ax2.grid(alpha=0.3)

# Plot 3: Top-5 Accuracy (val only)
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(epochs, va_top5, color='mediumorchid', label=f'Best: {max(va_top5):.1f}%')
ax3.axhline(max(va_top5), color='green', linestyle=':', alpha=0.7)
ax3.set(title='Val Top-5 Accuracy', xlabel='Epoch', ylabel='Accuracy (%)')
ax3.legend(); ax3.grid(alpha=0.3)

# Plot 4: LR schedule
ax4 = fig.add_subplot(gs[1, 0])
ax4.plot(epochs, lr_curve, color='darkorange')
ax4.set(title='Learning Rate Schedule', xlabel='Epoch', ylabel='LR')
ax4.grid(alpha=0.3)

# Plot 5: Val loss vs top1 scatter (generalisation view)
ax5 = fig.add_subplot(gs[1, 1])
sc  = ax5.scatter(va_loss, va_top1, c=epochs, cmap='viridis', s=20, alpha=0.8)
plt.colorbar(sc, ax=ax5, label='Epoch')
ax5.set(title='Val Loss vs Top-1 (generalisation)', xlabel='Val Loss', ylabel='Val Top-1 (%)')
ax5.grid(alpha=0.3)

# Plot 6: Train/Val top1 gap (overfitting indicator)
ax6 = fig.add_subplot(gs[1, 2])
gap = [t - v for t, v in zip(tr_top1, va_top1)]
ax6.fill_between(epochs, 0, gap, alpha=0.4, color='tomato')
ax6.plot(epochs, gap, color='tomato')
ax6.axhline(0, color='gray', linewidth=0.8)
ax6.set(title='Generalisation Gap (Train − Val Top-1)',
         xlabel='Epoch', ylabel='Gap (%)')
ax6.grid(alpha=0.3)

plt.suptitle('ASL Transformer — Training Summary', fontsize=14, fontweight='bold', y=1.01)
out_path = 'docs/thesis_figures/training_curves.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Saved: {out_path}')

## ⑨ Evaluation & Confusion Matrix

In [ ]:
import torch
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Load best checkpoint
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Loaded best checkpoint (epoch {ckpt["epoch"]}, val top-1: {ckpt["val_top1"]:.1f}%)')

# Collect predictions on test set
all_true, all_pred, all_conf = [], [], []
all_top5_correct = 0

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(DEVICE)
        logits = model(x)
        probs  = torch.softmax(logits, dim=-1)
        preds  = probs.argmax(dim=-1).cpu()
        confs  = probs.max(dim=-1).values.cpu()
        # top-5
        top5   = probs.topk(5, dim=1).indices.cpu()
        for yi, t5 in zip(y, top5):
            if yi.item() in t5.tolist(): all_top5_correct += 1
        all_true.extend(y.tolist())
        all_pred.extend(preds.tolist())
        all_conf.extend(confs.tolist())

top1_acc = np.mean(np.array(all_true) == np.array(all_pred)) * 100
top5_acc = all_top5_correct / len(all_true) * 100

print(f'\n═══ Test Set Results ═══')
print(f'Top-1 Accuracy : {top1_acc:.2f}%')
print(f'Top-5 Accuracy : {top5_acc:.2f}%')
print(f'Total samples  : {len(all_true)}')

# Class names
with open('data/raw/manifest.json') as f:
    mf = json.load(f)
i2w = {v: k for k, v in mf['word_to_idx'].items()}
class_names = [i2w[i] for i in range(NUM_CLASSES)]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(all_true, all_pred)

# Show the 30 words with most errors (cleaner for thesis)
errors     = cm.sum(axis=1) - np.diag(cm)
top_idx    = np.argsort(errors)[-30:]
cm_sub     = cm[np.ix_(top_idx, top_idx)]
sub_names  = [class_names[i] for i in top_idx]

fig, ax = plt.subplots(figsize=(16, 14))
im = ax.imshow(cm_sub, interpolation='nearest', cmap='Blues')
plt.colorbar(im)

ax.set(xticks=np.arange(30), yticks=np.arange(30),
       xticklabels=sub_names, yticklabels=sub_names,
       xlabel='Predicted', ylabel='True',
       title=f'Confusion Matrix — Top-30 Error Classes\n(Top-1: {top1_acc:.1f}%  Top-5: {top5_acc:.1f}%)')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

thresh = cm_sub.max() / 2
for i in range(30):
    for j in range(30):
        if cm_sub[i,j] > 0:
            ax.text(j, i, str(cm_sub[i,j]), ha='center', va='center', fontsize=6,
                    color='white' if cm_sub[i,j] > thresh else 'black')

plt.tight_layout()
out_path = 'docs/thesis_figures/confusion_matrix.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Saved: {out_path}')

In [ ]:
# Confidence distribution — correct vs wrong predictions
import matplotlib.pyplot as plt

correct = [c for t, p, c in zip(all_true, all_pred, all_conf) if t == p]
wrong   = [c for t, p, c in zip(all_true, all_pred, all_conf) if t != p]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(correct, bins=30, alpha=0.7, color='steelblue',  label=f'Correct (n={len(correct)})')
axes[0].hist(wrong,   bins=30, alpha=0.7, color='tomato',     label=f'Wrong   (n={len(wrong)})')
axes[0].set(title='Confidence Distribution', xlabel='Softmax Confidence', ylabel='Count')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Per-class top-1 accuracy (sorted)
per_class_correct = np.zeros(NUM_CLASSES)
per_class_total   = np.zeros(NUM_CLASSES)
for t, p in zip(all_true, all_pred):
    per_class_total[t]   += 1
    per_class_correct[t] += (t == p)
per_class_acc = np.where(per_class_total > 0,
                          per_class_correct / per_class_total * 100, 0)
sorted_idx = np.argsort(per_class_acc)
axes[1].barh([class_names[i] for i in sorted_idx[-20:]],
              per_class_acc[sorted_idx[-20:]], color='steelblue', alpha=0.8)
axes[1].set(title='Top-20 Best-Performing Signs', xlabel='Top-1 Accuracy (%)')
axes[1].grid(alpha=0.3, axis='x')

plt.tight_layout()
out_path = 'docs/thesis_figures/confidence_and_per_class.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Saved: {out_path}')

## ⑩ Ablation Study (Thesis Contribution)

In [ ]:
# Quick ablation: vary number of Transformer layers and measure val top-1
# Run this AFTER the main training to compare architecture choices

import torch, json, time
import torch.nn as nn
from torch.optim import AdamW

ABLATION_EPOCHS = 10   # short run for comparison
ablation_results = []

for n_layers in [1, 2, 4, 6]:
    print(f'\n--- Ablation: {n_layers} Transformer layers ---')
    abl_model = ASLTransformer(
        num_classes=NUM_CLASSES, num_layers=n_layers
    ).to(DEVICE)
    abl_opt   = AdamW(abl_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    abl_crit  = nn.CrossEntropyLoss(label_smoothing=0.1)

    best_va = 0
    for ep in range(1, ABLATION_EPOCHS + 1):
        run_epoch(abl_model, train_loader, abl_crit, abl_opt, DEVICE)
        va = run_epoch(abl_model, val_loader, abl_crit, None, DEVICE)
        if va['top1'] > best_va: best_va = va['top1']
        print(f'  ep {ep:2d}  val top1: {va["top1"]:5.1f}%')

    params = abl_model.count_parameters()
    ablation_results.append({'layers': n_layers, 'val_top1': best_va, 'params': params})
    print(f'  Best val top-1: {best_va:.1f}%  |  Params: {params:,}')
    del abl_model

print('\n═══ Ablation Summary ═══')
print(f'{"Layers":>8}  {"Val Top-1":>10}  {"Params":>10}')
for r in ablation_results:
    print(f'{r["layers"]:>8}  {r["val_top1"]:>9.1f}%  {r["params"]:>10,}')

In [ ]:
import matplotlib.pyplot as plt

layers    = [r['layers']    for r in ablation_results]
val_accs  = [r['val_top1']  for r in ablation_results]
params_m  = [r['params'] / 1e6 for r in ablation_results]

fig, ax1 = plt.subplots(figsize=(8, 5))
color1 = 'steelblue'
color2 = 'coral'

ax1.bar(layers, val_accs, color=color1, alpha=0.8, width=0.6, label='Val Top-1 Acc')
ax1.set(xlabel='Number of Transformer Layers', ylabel='Val Top-1 Accuracy (%)',
        title='Ablation: Transformer Depth vs Accuracy')
ax1.set_xticks(layers)

ax2 = ax1.twinx()
ax2.plot(layers, params_m, 'o--', color=color2, label='Model Params (M)')
ax2.set_ylabel('Parameters (M)', color=color2)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower right')
ax1.grid(alpha=0.3, axis='y')

plt.tight_layout()
out_path = 'docs/thesis_figures/ablation_layers.png'
plt.savefig(out_path, dpi=150)
plt.show()
print(f'✅ Saved: {out_path}')

## ⑪ Save Everything to Google Drive

In [ ]:
import shutil, os
from pathlib import Path

if GDRIVE_DIR:
    # Copy checkpoint
    shutil.copy(CKPT_PATH, f'{GDRIVE_DIR}/best_model.pth')

    # Copy thesis figures
    figs_src = Path('docs/thesis_figures')
    figs_dst = Path(f'{GDRIVE_DIR}/thesis_figures')
    figs_dst.mkdir(exist_ok=True)
    for fig_file in figs_src.glob('*.png'):
        shutil.copy(fig_file, figs_dst / fig_file.name)

    # Copy training history
    shutil.copy('training/logs/history.json', f'{GDRIVE_DIR}/history.json')

    # Summary JSON
    summary = {
        'test_top1': top1_acc,
        'test_top5': top5_acc,
        'best_val_top1': best_val_top1,
        'num_classes': NUM_CLASSES,
        'epochs_trained': len(history),
        'model_params': model.count_parameters(),
        'ablation': ablation_results if 'ablation_results' in dir() else [],
    }
    with open(f'{GDRIVE_DIR}/summary.json', 'w') as f:
        json.dump(summary, f, indent=2)

    print(f'✅ Everything saved to Google Drive: {GDRIVE_DIR}')
    print(f'   Files:')
    for p in sorted(Path(GDRIVE_DIR).rglob('*')):
        if p.is_file(): print(f'     {p.relative_to(GDRIVE_DIR)}')
else:
    print('⚠️  SAVE_TO_GDRIVE=False — files only saved locally in Colab (will be lost on disconnect)')
    print(f'   Checkpoint: {CKPT_PATH}')

## ⑫ Summary

After running all cells, you should have:

| Output | Location |
|---|---|
| Trained model | `models/checkpoints/best_model.pth` |
| Training curves | `docs/thesis_figures/training_curves.png` |
| Confusion matrix | `docs/thesis_figures/confusion_matrix.png` |
| Confidence dist. | `docs/thesis_figures/confidence_and_per_class.png` |
| Ablation study   | `docs/thesis_figures/ablation_layers.png` |
| Sample trajectory | `docs/thesis_figures/sample_trajectory.png` |
| Training log | `training/logs/history.json` |

### Next steps
1. Download `best_model.pth` and place it in `models/checkpoints/` locally
2. Run the FastAPI backend: `uvicorn backend.main:app --reload`
3. Run the React frontend: `cd frontend && npm install && npm run dev`
4. Open `http://localhost:3000` and test the live demo!

### Expected results (WLASL-100 state of the art)
| Metric | Target | SOTA |
|---|---|---|
| Top-1 | 55–65% | ~65–70% |
| Top-5 | 80–90% | ~85–92% |